# PyTorch básico: dos tensores a uma MLP para XOR

Este notebook é um laboratório guiado para quem está começando com PyTorch. A proposta é **executar a célula, observar o resultado e então modificar pequenos valores**.

Ao final, você será capaz de:

1. criar, inspecionar e transformar tensores;
2. distinguir operações elemento a elemento de multiplicação matricial;
3. compreender *broadcasting*, dispositivos e tipos numéricos;
4. acompanhar o cálculo de gradientes com `autograd` e a regra da cadeia;
5. reconhecer o ciclo `forward → loss → backward → step`;
6. construir e treinar uma MLP simples para aprender a função XOR.

> **Como usar:** execute as células na ordem. Nas células marcadas com `PRÁTICA`, altere apenas os valores sugeridos e execute novamente. Se algo ficar estranho, reinicie o kernel e use **Run All**.

> **Ambiente do projeto:** selecione o kernel **Python (env conda com Torch)**.

## 1. Preparação do ambiente

PyTorch representa dados e parâmetros com `torch.Tensor`. Nesta primeira célula importamos as bibliotecas, fixamos uma semente aleatória e escolhemos um dispositivo.

Uma semente torna os sorteios reproduzíveis: se escrevemos

$$
X \sim \mathcal{N}(0,1),
$$

os valores continuam aleatórios, mas a mesma semente produz a mesma sequência em cada execução. Em computadores Apple, o dispositivo pode ser `mps`; em máquinas com GPU NVIDIA, `cuda`; caso contrário, `cpu`.

**Exemplo:** um tensor criado diretamente em `device` já estará pronto para participar das operações seguintes sem cópias entre CPU e acelerador.

In [ ]:
import sys
import torch
from torch import nn
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo selecionado: {device}")

## 2. Tensores: os dados fundamentais do PyTorch

Um tensor é um arranjo multidimensional de números. O número de eixos é chamado de **ordem**, **rank** ou `ndim`:

$$
\begin{aligned}
\text{escalar} &: \quad x \in \mathbb{R} && (\text{ordem }0)\\
\text{vetor} &: \quad \mathbf{x} \in \mathbb{R}^{n} && (\text{ordem }1)\\
\text{matriz} &: \quad X \in \mathbb{R}^{m\times n} && (\text{ordem }2)\\
\text{tensor} &: \quad T \in \mathbb{R}^{d_1\times\cdots\times d_k} && (\text{ordem }k).
\end{aligned}
$$

Não existe um limite conceitual de três dimensões. Um lote de vídeos pode ter forma
$$(N, C, T, H, W),$$
isto é: exemplos, canais, quadros, altura e largura. Não conseguimos desenhar cinco eixos espaciais, mas conseguimos representar sua **forma** e selecionar fatias.

**Exemplo:** na célula seguinte, cada objeto ganha um eixo a mais.

In [ ]:
escalar = torch.tensor(7.0)
vetor = torch.tensor([1.0, 2.0, 3.0])
matriz = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]])
tensor_3d = torch.arange(24).reshape(2, 3, 4)
videos_5d = torch.zeros(2, 3, 4, 5, 6)  # N, C, T, H, W

objetos = {
    "escalar": escalar,
    "vetor": vetor,
    "matriz": matriz,
    "tensor 3D": tensor_3d,
    "lote de vídeos 5D": videos_5d,
}

for nome, tensor in objetos.items():
    print(f"{nome:19s} | shape={tuple(tensor.shape)!s:16s} | ndim={tensor.ndim} | numel={tensor.numel()}")

### 2.1 Forma, tipo e dispositivo

Três propriedades ajudam a entender qualquer tensor:

- `shape`: quantidade de elementos em cada eixo;
- `dtype`: representação numérica, como `float32` ou `int64`;
- `device`: onde os dados estão armazenados.

Para um tensor $X\in\mathbb{R}^{m\times n}$, `shape == (m, n)`. Redes neurais normalmente usam ponto flutuante porque seus parâmetros precisam variar continuamente. Índices e classes, por outro lado, frequentemente usam inteiros.

**Exemplo:** $X\in\mathbb{R}^{2\times3}$ possui seis elementos. A prática também mostra que uma operação exige tensores compatíveis em tipo e dispositivo.

In [ ]:
# PRÁTICA: troque dtype por torch.float64 e compare o número de bytes. Compatível apenas para CPU.

X = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)
classes = torch.tensor([0, 2, 1], dtype=torch.int64, device=device)

print("X:", X)
print("shape:", X.shape)
print("dtype:", X.dtype)
print("device:", X.device)
print("bytes aproximados:", X.numel() * X.element_size())
print("classes (índices inteiros):", classes)

### 2.2 Indexação e fatiamento

Indexar significa escolher partes do tensor. Em uma matriz $X$, `X[i, j]` seleciona um elemento; `X[i]` seleciona uma linha; `X[:, j]` seleciona uma coluna.

Se

$$
X=\begin{bmatrix}10&20&30\\40&50&60\\70&80&90\end{bmatrix},
$$

então $X_{1,2}=60$ ao usar índices iniciados em zero. Já `X[:2, 1:]` mantém as duas primeiras linhas e as colunas a partir da segunda.

**Exemplo e prática:** preveja a saída de cada seleção antes de executar.

In [ ]:
X = torch.tensor([[10, 20, 30],
                  [40, 50, 60],
                  [70, 80, 90]])

print("Elemento X[1, 2]:", X[1, 2].item())
print("Segunda linha:      ", X[1])
print("Terceira coluna:    ", X[:, 2])
print("Fatia X[:2, 1:]:\n", X[:2, 1:])

### 2.3 Operações elemento a elemento e multiplicação matricial

O operador `*` multiplica posições correspondentes:

$$
(A\odot B)_{ij}=A_{ij}B_{ij}.
$$

Já `@` realiza multiplicação matricial. Se $A\in\mathbb{R}^{m\times n}$ e $B\in\mathbb{R}^{n\times p}$, então

$$
C=AB,\qquad C_{ij}=\sum_{k=1}^{n}A_{ik}B_{kj},\qquad C\in\mathbb{R}^{m\times p}.
$$

**Exemplo:** uma camada linear usa exatamente esse padrão, $Y=XW+b$. Compare `A * A` com `A @ A` na saída.

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]], device=device)

produto_elemento = A * A
produto_matricial = A @ A

print("A * A (elemento a elemento):\n", produto_elemento)
print("\nA @ A (produto matricial):\n", produto_matricial)

# Checagem de uma entrada: C[0,0] = 1*1 + 2*3 = 7
assert produto_matricial[0, 0].item() == 7.0

### 2.4 Broadcasting

*Broadcasting* permite operar tensores de formas diferentes quando seus eixos são compatíveis. A comparação é feita da direita para a esquerda; cada par de dimensões deve ser igual ou uma delas deve valer 1.

Em uma camada linear, um mesmo viés é somado a todas as linhas do lote:

$$
Y_{ij}=X_{ij}+b_j,
$$

com $X\in\mathbb{R}^{N\times d}$ e $b\in\mathbb{R}^{d}$. O vetor $b$ é virtualmente repetido $N$ vezes, sem precisarmos escrever essa repetição.

**Exemplo:** some $b=[10,20,30]$ a cada uma das duas linhas de $X$.

In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]], device=device)
b = torch.tensor([10.0, 20.0, 30.0], device=device)
Y = X + b

print("shape de X:", tuple(X.shape))
print("shape de b:", tuple(b.shape))
print("X + b:\n", Y)

# PRÁTICA: experimente b = torch.tensor([10.0], device=device).

### 2.5 `reshape`, `unsqueeze` e `squeeze`

Mudar a forma não precisa mudar os dados. Se o número total de elementos for preservado,

$$
\prod_i d_i = \prod_j d'_j,
$$

podemos reorganizar um tensor de forma $(d_1,\ldots,d_k)$ para $(d'_1,\ldots,d'_r)$.

`unsqueeze(dim)` adiciona um eixo de tamanho 1; `squeeze(dim)` remove esse eixo. Isso é útil porque modelos esperam convenções específicas, como `(lote, atributos)`.

**Exemplo:** um vetor com quatro atributos, forma `(4,)`, torna-se um lote de um exemplo, forma `(1, 4)`.

In [ ]:
vetor = torch.arange(1, 5, dtype=torch.float32, device=device)
lote_de_um = vetor.unsqueeze(0)
coluna = vetor.unsqueeze(1)
matriz_2x2 = vetor.reshape(2, 2)

print("vetor:       ", vetor, vetor.shape)
print("lote de um:  ", lote_de_um, lote_de_um.shape)
print("vetor coluna:\n", coluna, coluna.shape)
print("matriz 2x2:\n", matriz_2x2, matriz_2x2.shape)
print("voltando com squeeze:", lote_de_um.squeeze(0).shape)

## 3. Autograd: derivadas automáticas

Quando `requires_grad=True`, PyTorch registra as operações aplicadas ao tensor em um grafo computacional dinâmico. Ao chamar `backward()`, ele percorre o grafo no sentido inverso e calcula derivadas.

Para

$$
z=x^2+3x,
$$

a derivada analítica é

$$
\frac{dz}{dx}=2x+3.
$$

**Exemplo:** em $x=2$, esperamos $dz/dx=7$. A propriedade `x.grad` deve confirmar esse valor.

In [ ]:
x = torch.tensor(2.0, requires_grad=True, device=device)
z = x**2 + 3*x

print("z =", z.item())
print("grad_fn de z:", z.grad_fn)

z.backward()
print("dz/dx calculado pelo autograd =", x.grad.item())
assert x.grad.item() == 7.0

### 3.1 Regra da cadeia: por que `w.grad = 8` e `b.grad = 4`?

Considere

$$
\hat y=wx+b,\qquad e=\hat y-y,\qquad L=e^2.
$$

Com $w=3$, $x=2$, $b=1$ e $y=5$, temos $\hat y=7$, $e=2$ e $L=4$. A regra da cadeia multiplica as derivadas locais ao longo de cada caminho:

$$
\frac{\partial L}{\partial w}
=\frac{\partial L}{\partial e}
 \frac{\partial e}{\partial \hat y}
 \frac{\partial \hat y}{\partial w}
=(2e)(1)(x)=(4)(1)(2)=8,
$$

$$
\frac{\partial L}{\partial b}
=\frac{\partial L}{\partial e}
 \frac{\partial e}{\partial \hat y}
 \frac{\partial \hat y}{\partial b}
=(2e)(1)(1)=4.
$$

**Prática:** execute, confira os gradientes e depois altere `x` ou `y`; antes de rodar novamente, tente prever os novos valores.

In [ ]:
w = torch.tensor(3.0, requires_grad=True, device=device)
b = torch.tensor(1.0, requires_grad=True, device=device)
x = torch.tensor(2.0, device=device)
y = torch.tensor(5.0, device=device)

y_hat = w*x + b
erro = y_hat - y
loss = erro**2
loss.backward()

print(f"y_hat={y_hat.item():.1f}, erro={erro.item():.1f}, loss={loss.item():.1f}")
print(f"w.grad={w.grad.item():.1f} (esperado: 8)")
print(f"b.grad={b.grad.item():.1f} (esperado: 4)")

assert torch.isclose(w.grad, torch.tensor(8.0, device=device))
assert torch.isclose(b.grad, torch.tensor(4.0, device=device))

### 3.2 Gradientes acumulam

Por padrão, PyTorch **soma** novos gradientes aos existentes:

$$
g_{\text{armazenado}} \leftarrow g_{\text{armazenado}} + \frac{\partial L}{\partial \theta}.
$$

Isso permite acumular contribuições de várias partes de um lote, mas também explica por que um treino deve limpar os gradientes antes de cada `backward()`.

**Exemplo:** para $L=w^2$ em $w=3$, cada chamada produz gradiente 6. Duas chamadas, sem zerar, deixam 12. Depois de `w.grad.zero_()`, o valor volta a zero.

In [ ]:
w = torch.tensor(3.0, requires_grad=True, device=device)

(w**2).backward()
print("Após o primeiro backward:", w.grad.item())

(w**2).backward()
print("Após o segundo backward: ", w.grad.item())

w.grad.zero_()
print("Após zerar o gradiente:   ", w.grad.item())

### 3.3 Inferência com `torch.no_grad()`

Durante treino precisamos de derivadas; durante avaliação, não. `torch.no_grad()` impede a construção do grafo, reduzindo uso de memória.

Se $\hat y=f_\theta(x)$, a inferência calcula apenas o valor de $f_\theta(x)$, sem pedir

$$
\nabla_\theta f_\theta(x).
$$

`detach()` também devolve um tensor desconectado do grafo. Para converter um tensor que pode estar em GPU/MPS para NumPy, o padrão seguro é `tensor.detach().cpu().numpy()`.

**Exemplo:** compare `grad_fn` dentro e fora do bloco de inferência.

In [ ]:
x = torch.tensor(2.0, requires_grad=True, device=device)
com_grafo = 3*x + 1

with torch.no_grad():
    sem_grafo = 3*x + 1

print("Com grafo: requires_grad =", com_grafo.requires_grad, "| grad_fn =", com_grafo.grad_fn)
print("Sem grafo: requires_grad =", sem_grafo.requires_grad, "| grad_fn =", sem_grafo.grad_fn)
print("Valor NumPy:", com_grafo.detach().cpu().numpy())

## 4. Primeiro aprendizado: regressão linear manual

Agora aplicaremos autograd a um problema de aprendizado. Geramos pontos próximos da reta

$$
y=2x+1+\varepsilon,\qquad \varepsilon\sim\mathcal{N}(0,\sigma^2),
$$

e pedimos ao modelo $\hat y=wx+b$ que descubra $w$ e $b$.

Usaremos o erro quadrático médio:

$$
\operatorname{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2.
$$

**Exemplo:** o conjunto é sintético e reproduzível; o gráfico deve parecer uma reta com pequeno ruído.

In [ ]:
torch.manual_seed(SEED)
x_train = torch.linspace(-1, 1, 80, device=device).unsqueeze(1)
ruido = 0.12 * torch.randn_like(x_train)
y_train = 2*x_train + 1 + ruido

plt.figure(figsize=(7, 3.6))
plt.scatter(x_train.cpu(), y_train.cpu(), s=22, alpha=0.75, label="dados")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados de treino: aproximadamente y = 2x + 1")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

### 4.1 Um passo de descida do gradiente

Para minimizar $L(w,b)$, movemos cada parâmetro na direção oposta ao gradiente:

$$
w \leftarrow w-\eta\frac{\partial L}{\partial w},\qquad
b \leftarrow b-\eta\frac{\partial L}{\partial b},
$$

onde $\eta$ é a taxa de aprendizado. O ciclo de um passo é:

1. `forward`: calcular $\hat y$;
2. `loss`: medir o erro;
3. `backward`: calcular gradientes;
4. atualizar parâmetros;
5. zerar gradientes.

**Exemplo:** começaremos com $w=0$ e $b=0$. A célula imprime a perda antes e depois de uma única atualização.

In [ ]:
w = torch.tensor(0.0, requires_grad=True, device=device)
b = torch.tensor(0.0, requires_grad=True, device=device)
taxa = 0.1

pred = w*x_train + b
loss = ((pred - y_train)**2).mean()
loss_inicial = loss.item()
loss.backward()

print(f"Antes: loss={loss_inicial:.4f}, dw={w.grad.item():.4f}, db={b.grad.item():.4f}")

with torch.no_grad():
    w -= taxa * w.grad
    b -= taxa * b.grad

w.grad.zero_()
b.grad.zero_()
nova_loss = ((w*x_train + b - y_train)**2).mean().item()
print(f"Depois de um passo: loss={nova_loss:.4f}, w={w.item():.4f}, b={b.item():.4f}")
assert nova_loss < loss_inicial

### 4.2 Repetindo o ciclo de otimização

Treinar significa repetir a atualização. Para épocas $t=0,1,\ldots,T-1$:

$$
\theta_{t+1}=\theta_t-\eta\nabla_\theta L_t,
\qquad \theta=(w,b).
$$

A curva de perda decrescente indica que o modelo está encontrando parâmetros melhores. Ao final, esperamos $w\approx2$ e $b\approx1$, mas não exatamente, pois adicionamos ruído.

**Prática:** altere `taxa` para `0.01`, `0.5` ou `2.0` e observe velocidade, oscilações ou divergência. Depois restaure `0.1`.

In [ ]:
torch.manual_seed(SEED)
w = torch.tensor(0.0, requires_grad=True, device=device)
b = torch.tensor(0.0, requires_grad=True, device=device)
taxa = 0.1
historico = []

for epoca in range(100):
    pred = w*x_train + b
    loss = ((pred - y_train)**2).mean()
    historico.append(loss.item())
    loss.backward()

    with torch.no_grad():
        w -= taxa * w.grad
        b -= taxa * b.grad

    w.grad.zero_()
    b.grad.zero_()

print(f"Parâmetros aprendidos: w={w.item():.3f}, b={b.item():.3f}")
print(f"Perda: {historico[0]:.4f} → {historico[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(historico, color="#087EAD")
axes[0].set(title="Perda durante o treino", xlabel="época", ylabel="MSE")
axes[0].grid(alpha=0.2)

with torch.no_grad():
    linha = w*x_train + b
axes[1].scatter(x_train.cpu(), y_train.cpu(), s=18, alpha=0.65, label="dados")
axes[1].plot(x_train.cpu(), linha.cpu(), color="#C73576", linewidth=2.5, label="modelo")
axes[1].set(title="Reta aprendida", xlabel="x", ylabel="y")
axes[1].grid(alpha=0.2)
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. A abstração `nn.Module`

Em vez de declarar e atualizar cada peso manualmente, definimos modelos como subclasses de `nn.Module`. Uma camada `nn.Linear(d_{in},d_{out})` calcula

$$
Y=XW^\top+b,
$$

onde $X\in\mathbb{R}^{N\times d_{in}}$, $W\in\mathbb{R}^{d_{out}\times d_{in}}$, $b\in\mathbb{R}^{d_{out}}$ e $Y\in\mathbb{R}^{N\times d_{out}}$.

`model.parameters()` reúne automaticamente todos os tensores treináveis. `nn.MSELoss()` implementa a perda e `torch.optim.SGD` implementa a atualização.

**Exemplo:** `nn.Linear(1, 1)` representa a mesma reta da seção anterior.

In [ ]:
torch.manual_seed(SEED)
modelo_linear = nn.Linear(in_features=1, out_features=1).to(device)

print(modelo_linear)
for nome, parametro in modelo_linear.named_parameters():
    print(f"{nome:12s} | shape={tuple(parametro.shape)} | requires_grad={parametro.requires_grad}")

amostra = torch.tensor([[0.5]], device=device)
print("Predição inicial para x=0.5:", modelo_linear(amostra).item())

### 5.1 Treino com perda e otimizador

O otimizador encapsula a regra de atualização. O ciclo padrão em PyTorch é:

$$
\hat y=f_\theta(X),\quad L=\ell(\hat y,y),\quad
\nabla_\theta L=\texttt{backward()},\quad
\theta\leftarrow\texttt{optimizer.step()}.
$$

`optimizer.zero_grad()` é chamado antes de `backward()` para evitar acúmulo não intencional. O `forward` não precisa ser chamado diretamente: `modelo_linear(x_train)` executa `__call__`, que organiza os mecanismos internos e então chama `forward`.

**Exemplo:** treinaremos a camada linear nos mesmos dados e conferiremos se a perda cai.

In [ ]:
criterio = nn.MSELoss()
otimizador = torch.optim.SGD(modelo_linear.parameters(), lr=0.1)
historico_module = []

for epoca in range(120):
    modelo_linear.train()
    pred = modelo_linear(x_train)       # forward
    loss = criterio(pred, y_train)      # loss

    otimizador.zero_grad()              # limpa gradientes antigos
    loss.backward()                     # backward
    otimizador.step()                    # atualiza os parâmetros
    historico_module.append(loss.item())

peso = modelo_linear.weight.item()
vies = modelo_linear.bias.item()
print(f"weight={peso:.3f}, bias={vies:.3f}, loss final={historico_module[-1]:.5f}")
assert historico_module[-1] < historico_module[0]

### 5.2 `train()`, `eval()` e inspeção dos gradientes

`model.train()` ativa o comportamento de treino; `model.eval()` ativa o de avaliação. Isso é essencial para camadas como *dropout* e *batch normalization*. Para uma camada linear simples, o resultado numérico não muda, mas adotar o padrão evita erros quando a arquitetura crescer.

Os gradientes têm a mesma forma dos parâmetros:

$$
\nabla_W L\in\mathbb{R}^{d_{out}\times d_{in}},
\qquad
\nabla_b L\in\mathbb{R}^{d_{out}}.
$$

**Exemplo:** faremos uma passada de avaliação sem grafo e, depois, uma passada de treino para observar `parametro.grad`.

In [ ]:
modelo_linear.eval()
with torch.no_grad():
    pred_avaliacao = modelo_linear(x_train[:3])
print("Três predições em avaliação:\n", pred_avaliacao.cpu())

modelo_linear.train()
otimizador.zero_grad()
loss = criterio(modelo_linear(x_train), y_train)
loss.backward()

for nome, parametro in modelo_linear.named_parameters():
    print(f"{nome:12s} | parâmetro={parametro.detach().cpu().numpy()} | grad={parametro.grad.detach().cpu().numpy()}")

## 6. MLP simples para aprender XOR

A função XOR vale 1 quando as duas entradas são diferentes:

| $x_1$ | $x_2$ | $y=x_1\oplus x_2$ |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Não existe uma única reta que separe os dois pontos positivos dos dois negativos. Por isso, um modelo puramente linear não resolve XOR. Uma MLP introduz uma transformação não linear:

$$
h=\tanh(XW_1^\top+b_1),\qquad
z=hW_2^\top+b_2,
$$

onde $z$ é um **logit**. A probabilidade é $p=\sigma(z)=1/(1+e^{-z})$.

**Exemplo:** construiremos o conjunto completo de quatro casos e o enviaremos ao dispositivo escolhido.

In [ ]:
X_xor = torch.tensor([[0, 0],
                      [0, 1],
                      [1, 0],
                      [1, 1]], dtype=torch.float32, device=device)
y_xor = torch.tensor([[0],
                      [1],
                      [1],
                      [0]], dtype=torch.float32, device=device)

print("X shape:", tuple(X_xor.shape), "| y shape:", tuple(y_xor.shape))
print(torch.cat([X_xor, y_xor], dim=1).cpu())

### 6.1 Arquitetura da MLP

Nossa rede terá duas entradas, quatro neurônios ocultos e uma saída:

$$
\mathbb{R}^{2}\xrightarrow{\text{Linear}(2,4)}
\mathbb{R}^{4}\xrightarrow{\tanh}
\mathbb{R}^{4}\xrightarrow{\text{Linear}(4,1)}
\mathbb{R}.
$$

A não linearidade é indispensável: sem `Tanh`, duas camadas lineares seguidas ainda equivaleriam a uma única transformação linear.

Usaremos `BCEWithLogitsLoss`, que combina sigmoide e entropia cruzada binária de forma estável:

$$
\ell(z,y)=-\left[y\log\sigma(z)+(1-y)\log(1-\sigma(z))\right].
$$

**Exemplo:** o `forward` devolve logits; a sigmoide será aplicada somente ao interpretar probabilidades.

In [ ]:
class XORMLP(nn.Module):
    def __init__(self, hidden_dim=4):
        super().__init__()
        self.layer1 = nn.Linear(2, hidden_dim)
        self.activation = nn.Tanh()
        self.layer2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = self.activation(self.layer1(x))
        logits = self.layer2(h)
        return logits


torch.manual_seed(SEED)
modelo_xor = XORMLP(hidden_dim=4).to(device)
print(modelo_xor)
print("Total de parâmetros treináveis:", sum(p.numel() for p in modelo_xor.parameters()))

logits_iniciais = modelo_xor(X_xor)
print("Shape dos logits:", tuple(logits_iniciais.shape))
print("Probabilidades iniciais:\n", torch.sigmoid(logits_iniciais).detach().cpu())

### 6.2 Treinando a MLP

Em cada época processamos os quatro exemplos, calculamos a perda, propagamos gradientes e atualizamos os parâmetros:

$$
\theta_{t+1}=\theta_t-\eta\,m_t,
$$

onde $m_t$ representa a direção adaptativa calculada pelo Adam a partir dos gradientes. Para este conjunto minúsculo, um único lote contém todos os exemplos.

Uma previsão binária usa o limiar $p\ge 0{,}5$. Em termos de logits, isso equivale a $z\ge0$, pois $\sigma(0)=0{,}5$.

**Prática:** depois da execução normal, experimente `hidden_dim=2`, `lr=0.01` ou menos épocas. Reinicialize o modelo antes de cada comparação para que o teste seja justo.

In [ ]:
criterio_xor = nn.BCEWithLogitsLoss()
otimizador_xor = torch.optim.Adam(modelo_xor.parameters(), lr=0.05)
historico_xor = []

for epoca in range(1500):
    modelo_xor.train()
    logits = modelo_xor(X_xor)
    loss = criterio_xor(logits, y_xor)

    otimizador_xor.zero_grad()
    loss.backward()
    otimizador_xor.step()

    historico_xor.append(loss.item())

    if epoca in {0, 99, 499, 999, 1499}:
        print(f"época {epoca + 1:4d} | loss={loss.item():.6f}")

plt.figure(figsize=(7, 3.5))
plt.plot(historico_xor, color="#087EAD")
plt.yscale("log")
plt.xlabel("época")
plt.ylabel("BCEWithLogitsLoss (escala log)")
plt.title("Aprendizado da função XOR")
plt.grid(alpha=0.2)
plt.show()

### 6.3 Avaliação: logits, probabilidades e classes

O modelo produz logits $z\in\mathbb{R}$. Para interpretar a saída:

$$
p(y=1\mid x)=\sigma(z),
\qquad
\hat y=\begin{cases}
1,&p\ge0{,}5,\\
0,&p<0{,}5.
\end{cases}
$$

Na avaliação usamos `eval()` e `torch.no_grad()`. Como o conjunto contém todos os casos possíveis, esperamos quatro acertos em quatro.

**Exemplo:** a tabela exibirá entrada, alvo, logit, probabilidade e classe prevista para cada linha.

In [ ]:
modelo_xor.eval()
with torch.no_grad():
    logits = modelo_xor(X_xor)
    probabilidades = torch.sigmoid(logits)
    classes = (probabilidades >= 0.5).float()
    acuracia = (classes == y_xor).float().mean()

print(" x1  x2 | alvo |  logit   | prob. | classe")
print("-" * 47)
for entrada, alvo, logit, prob, classe in zip(X_xor, y_xor, logits, probabilidades, classes):
    print(
        f" {int(entrada[0].item())}   {int(entrada[1].item())} |"
        f"   {int(alvo.item())}  | {logit.item():+8.3f} |"
        f" {prob.item():.3f} |   {int(classe.item())}"
    )

print(f"\nAcurácia: {acuracia.item():.0%}")

### 6.4 Visualizando a fronteira de decisão

A rede define uma probabilidade para qualquer ponto $(x_1,x_2)$ do plano:

$$
p(x_1,x_2)=\sigma\bigl(f_\theta(x_1,x_2)\bigr).
$$

A fronteira de decisão é a curva onde $p=0{,}5$. Uma reta não resolveria XOR, mas a camada oculta permite que a MLP construa uma região não linear.

**Exemplo:** o mapa de cores mostra $p(y=1)$; a linha branca marca $p=0{,}5$, e os quatro pontos são os exemplos de treino.

In [ ]:
eixo = torch.linspace(-0.25, 1.25, 180)
gx, gy = torch.meshgrid(eixo, eixo, indexing="xy")
grade = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1).to(device)

modelo_xor.eval()
with torch.no_grad():
    mapa_prob = torch.sigmoid(modelo_xor(grade)).reshape(gx.shape).cpu()

plt.figure(figsize=(6.2, 5))
contorno = plt.contourf(gx, gy, mapa_prob, levels=20, cmap="RdYlBu_r", vmin=0, vmax=1)
plt.contour(gx, gy, mapa_prob, levels=[0.5], colors="white", linewidths=2)
plt.scatter(
    X_xor[:, 0].cpu(), X_xor[:, 1].cpu(),
    c=y_xor.squeeze().cpu(), cmap="coolwarm", edgecolors="black", s=120
)
plt.colorbar(contorno, label="p(y=1)")
plt.xlabel("x₁")
plt.ylabel("x₂")
plt.title("Fronteira não linear aprendida para XOR")
plt.show()

## 7. Prática final e verificações

Uma execução correta deve satisfazer propriedades simples:

$$
\operatorname{shape}(f_\theta(X))=(4,1),\qquad
0\le p_i\le1,
$$

e, após o treino, a classe prevista deve coincidir com o alvo nos quatro casos de XOR.

As asserções abaixo funcionam como testes automáticos. Depois de confirmá-las, faça um experimento controlado: mude **uma variável por vez** (`hidden_dim`, ativação, taxa ou épocas), reinicie o kernel e compare a perda final e a acurácia.

**Desafio:** substitua `Tanh` por `ReLU`; formule uma hipótese antes de executar. A rede sempre converge com a mesma semente? O que muda ao reduzir a camada oculta para dois neurônios?

In [ ]:
assert logits.shape == (4, 1), "A saída deve ter um logit por exemplo."
assert torch.all((probabilidades >= 0) & (probabilidades <= 1)), "Probabilidades devem estar entre 0 e 1."
assert torch.equal(classes, y_xor), "A MLP ainda não aprendeu os quatro casos de XOR."
assert historico_xor[-1] < historico_xor[0], "A perda deveria diminuir."

print("✓ Formas corretas")
print("✓ Probabilidades no intervalo [0, 1]")
print("✓ Perda diminuiu durante o treino")
print("✓ Os quatro casos de XOR foram classificados corretamente")

## 8. Síntese e próximos passos

O percurso completo pode ser resumido por

$$
\text{tensor} \rightarrow \text{forward} \rightarrow \text{loss}
\rightarrow \text{backward} \rightarrow \text{step}.
$$

- **Tensores** carregam dados e parâmetros, inclusive estruturas com muitas dimensões.
- **Autograd** aplica a regra da cadeia para preencher `.grad`.
- **`nn.Module`** organiza camadas e parâmetros.
- **A função de perda** define o que significa errar.
- **O otimizador** usa os gradientes para atualizar os parâmetros.
- **A não linearidade** permite à MLP aprender relações que uma reta não representa, como XOR.

Próximas extensões sugeridas:

1. separar dados em treino e teste;
2. usar `Dataset` e `DataLoader` para criar minilotes;
3. estudar classificação multiclasse com `CrossEntropyLoss`;
4. treinar uma MLP em MNIST ou Fashion-MNIST;
5. salvar e restaurar pesos com `state_dict`.

> Pergunta de fechamento: se `loss.backward()` calcula os gradientes, qual linha realmente altera os pesos? Resposta: `optimizer.step()`.